# EDA bộ dữ liệu ViVoice34

Notebook phân tích bộ dữ liệu [`anonymous-vivoice34/ViVoice34`](https://huggingface.co/datasets/anonymous-vivoice34/ViVoice34) trên Hugging Face, tập trung vào việc tìm giọng Việt đa dạng theo tỉnh/vùng miền để làm dữ liệu tham chiếu TTS.

Mục tiêu:

- đọc **metadata trước**, không tải toàn bộ audio gần 30 GB;
- kiểm tra quy mô, thiếu dữ liệu, trùng lặp và tính nhất quán;
- phân tích tỉnh, vùng suy diễn, giới tính, nhóm tuổi, chủ đề và người nói;
- phân tích độ dài audio, transcript và tốc độ nói gần đúng;
- tải có kiểm soát một vài audio để nghe, xem waveform/spectrogram và đo chất lượng tín hiệu;
- xuất metadata đã làm giàu cùng shortlist giọng tham chiếu.

> Dataset card ghi giấy phép **CC BY 4.0**. Tuy vậy, giấy phép dữ liệu không tự động thay thế yêu cầu về quyền nhân thân hoặc sự đồng ý khi clone giọng. Chỉ sử dụng giọng khi mục đích và quyền sử dụng phù hợp; công khai nội dung do AI tạo khi cần.

In [ ]:
%pip install -q "datasets>=3.0" "huggingface_hub>=0.25" pyarrow pandas numpy matplotlib seaborn soundfile librosa requests

## 1. Cấu hình và thư viện

`LOAD_AUDIO_SAMPLES=False` giúp chạy toàn bộ EDA metadata nhanh và nhẹ. Đổi thành `True` khi muốn tải một số audio để nghe và kiểm tra tín hiệu.

In [ ]:
from __future__ import annotations

import io
import os
import re
import unicodedata
from itertools import islice
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from IPython.display import Audio, Markdown, display
from huggingface_hub import HfFileSystem

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.unicode_minus"] = False

DATASET_ID = "anonymous-vivoice34/ViVoice34"
SPLIT = "train"
RANDOM_SEED = 42
OUTPUT_DIR = Path("/content/vivoice34_eda")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOAD_AUDIO_SAMPLES = False
AUDIO_OFFSET = 0
AUDIO_SAMPLE_SIZE = 4
PREVIEW_SECONDS = 12

print("Dataset :", DATASET_ID)
print("Output  :", OUTPUT_DIR.resolve())
print("Audio EDA:", "bật" if LOAD_AUDIO_SAMPLES else "tắt")

## 2. Đọc metadata mà không tải cột audio

Các file Parquet lưu audio nhúng có thể rất lớn. Cell dưới chỉ đọc những cột metadata cần thiết bằng column projection. Với dataset riêng tư/gated, đặt token vào biến môi trường `HF_TOKEN`; dataset hiện tại là công khai nên thường không cần.

In [ ]:
fs = HfFileSystem(token=os.getenv("HF_TOKEN"))
repo_prefix = f"datasets/{DATASET_ID}"
parquet_paths = sorted(fs.glob(f"{repo_prefix}/**/*.parquet"))

if not parquet_paths:
    raise FileNotFoundError(
        f"Không tìm thấy Parquet trong {DATASET_ID}. "
        "Hãy kiểm tra kết nối mạng, tên dataset hoặc HF_TOKEN."
    )

print(f"Tìm thấy {len(parquet_paths)} file Parquet:")
for path in parquet_paths:
    print(" -", path)

In [ ]:
requested_columns = [
    "file_name",
    "transcript",
    "speaker_id",
    "province",
    "gender",
    "age_group",
    "duration_s",
    "total_word",
    "province_code",
]

storage_options = {"token": os.getenv("HF_TOKEN")} if os.getenv("HF_TOKEN") else None
metadata_frames = []

for path in parquet_paths:
    url = f"hf://{path}"
    frame = pd.read_parquet(
        url,
        columns=requested_columns,
        storage_options=storage_options,
    )
    frame["source_parquet"] = path.rsplit("/", 1)[-1]
    metadata_frames.append(frame)

df = pd.concat(metadata_frames, ignore_index=True)
df.insert(0, "row_id", np.arange(len(df), dtype=int))

for column in ["file_name", "transcript", "speaker_id", "province", "gender", "age_group", "province_code"]:
    df[column] = df[column].astype("string").str.strip()
for column in ["duration_s", "total_word"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print(f"Đã đọc {len(df):,} dòng × {df.shape[1]} cột metadata.")
display(df.head())

## 3. Làm giàu metadata

Dataset không công bố cột vùng miền trong schema hiện tại. `derived_region` dưới đây là nhãn địa lý vĩ mô được suy diễn từ tên tỉnh để phục vụ EDA, **không phải nhãn gốc của dataset**. Các tỉnh chưa ánh xạ được giữ là `Chưa ánh xạ`.

In [ ]:
REGION_BY_PROVINCE = {
    # Miền Bắc
    "Hà Nội": "Bắc", "Hải Phòng": "Bắc", "Quảng Ninh": "Bắc",
    "Cao Bằng": "Bắc", "Lạng Sơn": "Bắc", "Bắc Ninh": "Bắc",
    "Bắc Giang": "Bắc", "Hưng Yên": "Bắc", "Hải Dương": "Bắc",
    "Ninh Bình": "Bắc", "Nam Định": "Bắc", "Hà Nam": "Bắc",
    "Thái Bình": "Bắc", "Phú Thọ": "Bắc", "Vĩnh Phúc": "Bắc",
    "Tuyên Quang": "Bắc", "Hà Giang": "Bắc", "Lào Cai": "Bắc",
    "Yên Bái": "Bắc", "Lai Châu": "Bắc", "Điện Biên": "Bắc",
    "Sơn La": "Bắc", "Hòa Bình": "Bắc", "Thái Nguyên": "Bắc",
    "Bắc Kạn": "Bắc",
    # Miền Trung và Tây Nguyên
    "Thanh Hóa": "Trung", "Nghệ An": "Trung", "Hà Tĩnh": "Trung",
    "Quảng Bình": "Trung", "Quảng Trị": "Trung", "Thừa Thiên Huế": "Trung",
    "Huế": "Trung", "Đà Nẵng": "Trung", "Quảng Nam": "Trung",
    "Quảng Ngãi": "Trung", "Bình Định": "Trung", "Phú Yên": "Trung",
    "Khánh Hòa": "Trung", "Ninh Thuận": "Trung", "Bình Thuận": "Trung",
    "Kon Tum": "Trung", "Gia Lai": "Trung", "Đắk Lắk": "Trung",
    "Đắk Nông": "Trung", "Lâm Đồng": "Trung",
    # Miền Nam
    "TP. Hồ Chí Minh": "Nam", "Hồ Chí Minh": "Nam", "Bà Rịa - Vũng Tàu": "Nam",
    "Đồng Nai": "Nam", "Bình Dương": "Nam", "Bình Phước": "Nam",
    "Tây Ninh": "Nam", "Long An": "Nam", "Tiền Giang": "Nam",
    "Bến Tre": "Nam", "Trà Vinh": "Nam", "Vĩnh Long": "Nam",
    "Đồng Tháp": "Nam", "An Giang": "Nam", "Kiên Giang": "Nam",
    "Cần Thơ": "Nam", "Hậu Giang": "Nam", "Sóc Trăng": "Nam",
    "Bạc Liêu": "Nam", "Cà Mau": "Nam",
}

def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFC", str(value)).strip()
    return re.sub(r"\s+", " ", text)

df["transcript_clean"] = df["transcript"].map(normalize_text)
df["derived_region"] = df["province"].map(REGION_BY_PROVINCE).fillna("Chưa ánh xạ")
df["transcript_chars"] = df["transcript_clean"].str.len()
df["word_count_calc"] = df["transcript_clean"].str.split().str.len()
df["words_per_minute"] = np.where(
    df["duration_s"].gt(0),
    60 * df["word_count_calc"] / df["duration_s"],
    np.nan,
)
df["has_terminal_punctuation"] = df["transcript_clean"].str.contains(r"[.!?…][\"'’”)]?$", regex=True)
df["is_duplicate_file"] = df["file_name"].duplicated(keep=False)
df["is_duplicate_transcript"] = df["transcript_clean"].duplicated(keep=False)

unmapped = sorted(df.loc[df["derived_region"].eq("Chưa ánh xạ"), "province"].dropna().unique())
print("Tỉnh chưa ánh xạ:", unmapped if unmapped else "Không có")
display(df.head())

## 4. Tổng quan và chất lượng metadata

In [ ]:
summary = pd.DataFrame(
    {
        "Chỉ số": [
            "Số bản ghi", "Số speaker", "Số tỉnh", "Tổng thời lượng (giờ)",
            "Thời lượng trung vị (giây)", "Transcript duy nhất",
            "File trùng", "Transcript thuộc nhóm trùng",
        ],
        "Giá trị": [
            len(df),
            df["speaker_id"].nunique(dropna=True),
            df["province"].nunique(dropna=True),
            round(df["duration_s"].sum() / 3600, 2),
            round(df["duration_s"].median(), 2),
            df["transcript_clean"].nunique(dropna=True),
            int(df["is_duplicate_file"].sum()),
            int(df["is_duplicate_transcript"].sum()),
        ],
    }
)
display(summary)

missing = (
    df.isna().sum()
    .to_frame("missing_count")
    .assign(missing_percent=lambda x: (100 * x["missing_count"] / len(df)).round(2))
    .sort_values(["missing_count", "missing_percent"], ascending=False)
)
display(Markdown("### Giá trị thiếu"))
display(missing)

display(Markdown("### Số giá trị duy nhất"))
display(df.nunique(dropna=True).sort_values().to_frame("n_unique"))

In [ ]:
quality_checks = pd.Series(
    {
        "duration_missing_or_nonpositive": int((df["duration_s"].isna() | df["duration_s"].le(0)).sum()),
        "empty_transcript": int(df["transcript_clean"].eq("").sum()),
        "missing_speaker_id": int(df["speaker_id"].isna().sum()),
        "duplicate_file_name_rows": int(df["is_duplicate_file"].sum()),
        "speaker_in_multiple_provinces": int((df.groupby("speaker_id")["province"].nunique() > 1).sum()),
        "speaker_in_multiple_genders": int((df.groupby("speaker_id")["gender"].nunique() > 1).sum()),
        "word_count_disagrees_with_total_word": int(
            (df["total_word"].notna() & df["word_count_calc"].ne(df["total_word"])).sum()
        ),
        "missing_terminal_punctuation": int((~df["has_terminal_punctuation"]).sum()),
    },
    name="count",
).to_frame()
display(quality_checks)

## 5. Phân bố tỉnh, vùng, giới tính, nhóm tuổi và chủ đề

In [ ]:
province_stats = (
    df.groupby(["derived_region", "province"], dropna=False)
    .agg(
        records=("row_id", "size"),
        speakers=("speaker_id", "nunique"),
        duration_hours=("duration_s", lambda s: s.sum() / 3600),
        median_duration_s=("duration_s", "median"),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)
display(province_stats.style.format({"duration_hours": "{:.2f}", "median_duration_s": "{:.1f}"}))

fig, axes = plt.subplots(1, 2, figsize=(17, max(6, 0.34 * len(province_stats))))
sns.barplot(data=province_stats, y="province", x="records", hue="derived_region", dodge=False, ax=axes[0])
axes[0].set(title="Số bản ghi theo tỉnh", xlabel="Số bản ghi", ylabel="Tỉnh")
sns.barplot(data=province_stats, y="province", x="duration_hours", hue="derived_region", dodge=False, ax=axes[1])
axes[1].set(title="Thời lượng theo tỉnh", xlabel="Giờ", ylabel="")
for ax in axes:
    ax.legend(title="Vùng suy diễn")
plt.tight_layout()
plt.show()

In [ ]:
categorical_columns = ["derived_region", "gender", "age_group", "province_code"]
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for column, ax in zip(categorical_columns, axes.flat):
    counts = df[column].fillna("Thiếu").value_counts().rename_axis(column).reset_index(name="count")
    sns.barplot(data=counts, x="count", y=column, ax=ax, color="#4C78A8")
    ax.set_title(f"Phân bố {column}")
    ax.set_xlabel("Số bản ghi")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
province_gender = pd.crosstab(df["province"], df["gender"])
province_gender = province_gender.loc[province_gender.sum(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(10, max(6, 0.38 * len(province_gender))))
sns.heatmap(province_gender, annot=True, fmt="g", cmap="Blues", cbar=False)
plt.title("Ma trận số bản ghi: tỉnh × giới tính")
plt.xlabel("Giới tính")
plt.ylabel("Tỉnh")
plt.tight_layout()
plt.show()

## 6. Độ dài audio và transcript

In [ ]:
display(
    df[["duration_s", "word_count_calc", "transcript_chars", "words_per_minute"]]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .T
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.histplot(df["duration_s"], bins=25, kde=True, ax=axes[0, 0], color="#4C78A8")
axes[0, 0].set(title="Phân bố thời lượng", xlabel="Giây")
sns.boxplot(data=df, x="derived_region", y="duration_s", ax=axes[0, 1], color="#72B7B2")
axes[0, 1].set(title="Thời lượng theo vùng suy diễn", xlabel="Vùng", ylabel="Giây")
sns.histplot(df["word_count_calc"], bins=25, kde=True, ax=axes[1, 0], color="#F58518")
axes[1, 0].set(title="Phân bố số từ transcript", xlabel="Số từ")
sns.histplot(df["words_per_minute"].replace([np.inf, -np.inf], np.nan).dropna(), bins=25, kde=True, ax=axes[1, 1], color="#E45756")
axes[1, 1].set(title="Tốc độ nói gần đúng", xlabel="Từ/phút")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="duration_s",
    y="word_count_calc",
    hue="derived_region",
    style="gender",
    alpha=0.8,
)
plt.title("Quan hệ giữa thời lượng và số từ")
plt.xlabel("Thời lượng (giây)")
plt.ylabel("Số từ tính lại")
plt.tight_layout()
plt.show()

wpm_bounds = df["words_per_minute"].quantile([0.01, 0.99])
wpm_outliers = df[
    df["words_per_minute"].lt(wpm_bounds.loc[0.01])
    | df["words_per_minute"].gt(wpm_bounds.loc[0.99])
][["row_id", "speaker_id", "province", "duration_s", "word_count_calc", "words_per_minute", "transcript_clean"]]
display(Markdown("### Các điểm ngoài khoảng phân vị 1%–99% của tốc độ nói"))
display(wpm_outliers.sort_values("words_per_minute"))

## 7. Người nói và transcript trùng

Transcript trùng không nhất thiết là lỗi: nhiều người cùng đọc một kịch bản có thể hữu ích cho so sánh giọng. Tuy nhiên cần nhận biết điều này trước khi chia train/validation/test để tránh rò rỉ nội dung.

In [ ]:
speaker_stats = (
    df.groupby("speaker_id", dropna=False)
    .agg(
        records=("row_id", "size"),
        total_minutes=("duration_s", lambda s: s.sum() / 60),
        province=("province", lambda s: " | ".join(sorted(s.dropna().unique()))),
        gender=("gender", lambda s: " | ".join(sorted(s.dropna().unique()))),
    )
    .reset_index()
    .sort_values(["records", "total_minutes"], ascending=False)
)
display(speaker_stats.head(20).style.format({"total_minutes": "{:.2f}"}))

plt.figure(figsize=(10, 5))
sns.histplot(speaker_stats["records"], discrete=True)
plt.title("Số bản ghi trên mỗi speaker")
plt.xlabel("Số bản ghi")
plt.ylabel("Số speaker")
plt.tight_layout()
plt.show()

In [ ]:
duplicate_transcripts = (
    df.groupby("transcript_clean", dropna=False)
    .agg(
        records=("row_id", "size"),
        speakers=("speaker_id", "nunique"),
        provinces=("province", lambda s: " | ".join(sorted(s.dropna().unique()))),
        sample_text=("transcript_clean", "first"),
    )
    .query("records > 1")
    .sort_values("records", ascending=False)
    .reset_index(drop=True)
)
display(duplicate_transcripts.head(20))

## 8. Shortlist giọng để nghe thử

Shortlist lấy tối đa một bản ghi cho mỗi tổ hợp vùng–tỉnh–giới tính, ưu tiên clip có tốc độ nói gần trung vị và metadata đầy đủ. Audio gốc dài 20–113 giây nên vẫn cần nghe, kiểm tra nhiễu và cắt một đoạn sạch 3–10 giây trước khi dùng làm reference cho F5-TTS.

In [ ]:
median_wpm = df["words_per_minute"].median()
candidate_pool = df.copy()
candidate_pool["wpm_distance"] = (candidate_pool["words_per_minute"] - median_wpm).abs()
candidate_pool["metadata_complete"] = candidate_pool[
    ["speaker_id", "province", "gender", "duration_s", "transcript_clean"]
].notna().all(axis=1)

shortlist = (
    candidate_pool
    .sort_values(["metadata_complete", "wpm_distance", "duration_s"], ascending=[False, True, True])
    .groupby(["derived_region", "province", "gender"], dropna=False, as_index=False)
    .head(1)
    [[
        "row_id", "speaker_id", "derived_region", "province", "gender", "age_group",
        "province_code", "duration_s", "word_count_calc", "words_per_minute",
        "file_name", "transcript_clean",
    ]]
    .sort_values(["derived_region", "province", "gender"])
    .reset_index(drop=True)
)
display(shortlist)

## 9. EDA audio có kiểm soát (tùy chọn)

Cell này dùng streaming và `Audio(decode=False)` để chỉ lấy `AUDIO_SAMPLE_SIZE` bản ghi kể từ `AUDIO_OFFSET`. Không chuyển dataset thành `pandas` và không tải toàn bộ audio. Nếu chỉ cần EDA metadata, giữ `LOAD_AUDIO_SAMPLES=False`.

In [ ]:
def decode_audio_object(audio_obj: dict) -> tuple[np.ndarray, int]:
    """Đọc audio từ bytes hoặc path/URL trả về bởi datasets với decode=False."""
    payload = audio_obj.get("bytes")
    path = audio_obj.get("path")

    if payload is not None:
        signal, sample_rate = sf.read(io.BytesIO(payload), dtype="float32", always_2d=True)
    elif path:
        signal, sample_rate = sf.read(path, dtype="float32", always_2d=True)
    else:
        raise ValueError("Audio không có cả bytes lẫn path.")

    mono = signal.mean(axis=1)
    return mono, int(sample_rate)


def audio_metrics(signal: np.ndarray, sample_rate: int) -> dict:
    eps = np.finfo(np.float32).eps
    peak = float(np.max(np.abs(signal))) if signal.size else 0.0
    rms = float(np.sqrt(np.mean(np.square(signal)))) if signal.size else 0.0
    frame_rms = librosa.feature.rms(y=signal, frame_length=2048, hop_length=512)[0]
    frame_db = librosa.amplitude_to_db(np.maximum(frame_rms, eps), ref=1.0)
    return {
        "sample_rate": sample_rate,
        "decoded_duration_s": len(signal) / sample_rate,
        "peak_dbfs": 20 * np.log10(max(peak, eps)),
        "rms_dbfs": 20 * np.log10(max(rms, eps)),
        "clipping_percent": 100 * float(np.mean(np.abs(signal) >= 0.999)),
        "quiet_frame_percent": 100 * float(np.mean(frame_db < -40)),
    }


audio_examples = []
audio_quality = pd.DataFrame()

if LOAD_AUDIO_SAMPLES:
    from datasets import Audio as HFAudio
    from datasets import load_dataset

    stream = load_dataset(DATASET_ID, split=SPLIT, streaming=True, token=os.getenv("HF_TOKEN"))
    stream = stream.cast_column("audio", HFAudio(decode=False))
    sampled_rows = list(islice(iter(stream), AUDIO_OFFSET, AUDIO_OFFSET + AUDIO_SAMPLE_SIZE))

    metric_rows = []
    for offset, row in enumerate(sampled_rows, start=AUDIO_OFFSET):
        signal, sample_rate = decode_audio_object(row["audio"])
        metric_rows.append({
            "stream_row": offset,
            "speaker_id": row.get("speaker_id"),
            "province": row.get("province"),
            "gender": row.get("gender"),
            **audio_metrics(signal, sample_rate),
        })
        audio_examples.append({
            "row": row,
            "signal": signal[: sample_rate * PREVIEW_SECONDS],
            "sample_rate": sample_rate,
        })

    audio_quality = pd.DataFrame(metric_rows)
    display(audio_quality)
else:
    print("Audio EDA đang tắt. Đổi LOAD_AUDIO_SAMPLES = True ở phần cấu hình rồi chạy lại cell này.")

In [ ]:
if audio_examples:
    for example in audio_examples:
        row = example["row"]
        signal = example["signal"]
        sample_rate = example["sample_rate"]

        display(Markdown(f"### {row.get('speaker_id')} · {row.get('province')} · {row.get('gender')}"))
        display(Markdown(f"**Transcript:** {row.get('transcript', '')}"))
        display(Audio(signal, rate=sample_rate))

        fig, axes = plt.subplots(1, 2, figsize=(16, 4))
        librosa.display.waveshow(signal, sr=sample_rate, ax=axes[0])
        axes[0].set(title=f"Waveform — {PREVIEW_SECONDS} giây đầu", xlabel="Thời gian (s)")

        mel = librosa.feature.melspectrogram(y=signal, sr=sample_rate, n_mels=80, fmax=min(8000, sample_rate // 2))
        mel_db = librosa.power_to_db(mel, ref=np.max)
        image = librosa.display.specshow(mel_db, sr=sample_rate, x_axis="time", y_axis="mel", ax=axes[1])
        axes[1].set(title="Mel spectrogram")
        fig.colorbar(image, ax=axes[1], format="%+2.0f dB")
        plt.tight_layout()
        plt.show()
else:
    print("Chưa có audio mẫu để hiển thị.")

## 10. Xuất kết quả

Hai file CSV được lưu trong `OUTPUT_DIR`:

- `vivoice34_metadata_enriched.csv`: metadata đã làm sạch và bổ sung biến EDA;
- `vivoice34_reference_shortlist.csv`: danh sách ngắn để nghe và chọn reference.

In [ ]:
metadata_output = OUTPUT_DIR / "vivoice34_metadata_enriched.csv"
shortlist_output = OUTPUT_DIR / "vivoice34_reference_shortlist.csv"

df.to_csv(metadata_output, index=False, encoding="utf-8-sig")
shortlist.to_csv(shortlist_output, index=False, encoding="utf-8-sig")

print("Đã lưu:")
print(" -", metadata_output)
print(" -", shortlist_output)

## Gợi ý đọc kết quả

Khi chọn giọng tham chiếu cho F5-TTS, ưu tiên:

1. vùng/tỉnh và giới tính còn thiếu trong bộ giọng hiện có;
2. một người nói duy nhất, phát âm rõ, không nhạc nền hoặc tiếng nói chồng;
3. peak không chạm 0 dBFS, clipping gần 0%, khoảng lặng vừa phải;
4. cắt đoạn liên tục 3–10 giây và chép transcript đúng từng chữ;
5. thử cùng một tập câu chuẩn để so sánh độ tự nhiên, độ giữ giọng và độ giữ vùng miền.

Không kết luận chất lượng giọng chỉ từ metadata: bước nghe thủ công vẫn bắt buộc.